# 🥈 Capa Silver: Jobs Data Engineering

## Objetivo
Transformar y limpiar los datos de la capa Bronze para crear una tabla Silver lista para análisis y consumo.

## 📦 Arquitectura con Tablas Temporales

**¿Por qué usar vistas temporales?**
* ✅ **Separación de lógica**: Las transformaciones están en la vista temporal, la persistencia en la tabla final
* ✅ **Testing más fácil**: Puedes validar la vista temporal antes de persistir
* ✅ **Reutilización**: La vista temporal puede ser consultada múltiples veces
* ✅ **Mejor rendimiento**: Las optimizaciones se aplican en la vista temporal

**Flujo:**
1. 📥 **Celda 2**: `CREATE TEMPORARY VIEW temp_silver_transformed` con todas las transformaciones
2. 📦 **Celda 3**: `CREATE TABLE jobs_silver` desde la vista temporal + timestamp

## Transformaciones Aplicadas

### ✅ Limpieza de Datos
* Eliminación de duplicados basado en `job_id`
* Normalización de campos de ubicación
* Manejo de valores nulos

### ✅ Tipos de Datos Correctos
* `job_is_remote`: BOOLEAN
* `job_apply_is_direct`: BOOLEAN
* `job_salary`: DOUBLE
* Coordenadas: DOUBLE
* Fechas: preparadas para parsing

### ✅ Campos Derivados
* `ubicacion_completa`: Concatenación de país y estado
* Extracción de información salarial estructurada

---

**Fuente:** `prueba_api.bronze.jobs_bronze`  
**Destino:** `prueba_api.silver.jobs_silver`  
**Registros esperados:** 28 (eliminando 1 duplicado)

In [0]:

-- PASO 1: CREAR VISTA TEMPORAL CON TRANSFORMACIONES


CREATE OR REPLACE TEMPORARY VIEW temp_silver_transformed AS
WITH deduplicated_data AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (PARTITION BY job_id ORDER BY job_posted_at_timestamp DESC) AS rn
  FROM prueba_api.bronze.jobs_bronze
),
cleaned_data AS (
  SELECT
    job_id,
    employer_name,
    employer_logo,
    employer_website,
    employer_reviews,
    job_title,
    job_description,
    job_publisher,
    
    -- ✅ NORMALIZACIÓN DE job_employment_type: Estandarizar a inglés
    CASE
      -- Full-time (con todas sus variantes y guiones)
      WHEN job_employment_type IN ('Full-time', 'Full–time', 'Tiempo completo', 'Tempo integral') 
        THEN 'Full-time'
      -- Part-time
      WHEN job_employment_type IN ('Part-time', 'Medio tiempo') 
        THEN 'Part-time'
      -- Contractor
      WHEN job_employment_type IN ('Contractor', 'Contratista', 'Prestador de serviços') 
        THEN 'Contractor'
      -- Internship
      WHEN job_employment_type IN ('Internship', 'Pasantía', 'Estágio') 
        THEN 'Internship'
      -- Combinaciones (normalizar guiones)
      WHEN job_employment_type IN ('Full-time and Part-time', 'Full–time and Part-time') 
        THEN 'Full-time and Part-time'
      WHEN job_employment_type IN ('Full-time and Contractor', 'Full–time and Contractor') 
        THEN 'Full-time and Contractor'
      WHEN job_employment_type IN ('Full-time, Part-time, and Contractor', 'Full–time, Part-time and Contractor', 'Full–time, Part-time, and Contractor') 
        THEN 'Full-time, Part-time, and Contractor'
      ELSE job_employment_type
    END AS job_employment_type,
    
    job_employment_types,
    
    CASE
      WHEN job_location IN ('Qualquer lugar', 'Anywhere') THEN 'Remote'
      ELSE TRIM(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(job_location, '\\s*[•·]\\s*$', ''), '\\s+\\(\\+\\d+.*\\)\\s*$', ''), '\\s+\\(y \\d+.*\\)\\s*$', ''))
    END AS job_location,
    
    job_city,
    job_state,
    
    CASE 
      WHEN job_country = 'CA' THEN 'Canada'
      WHEN job_country = 'US' THEN 'United States'
      WHEN job_country = 'MX' THEN 'Mexico'
      WHEN job_country = 'BR' THEN 'Brazil'
      WHEN job_country = 'CO' THEN 'Colombia'
      WHEN job_country = 'CL' THEN 'Chile'
      WHEN job_country = 'AR' THEN 'Argentina'
      WHEN job_country = 'IN' THEN 'India'
      ELSE job_country
    END AS job_country,
    
    CONCAT(
      CASE 
        WHEN job_country = 'CA' THEN 'CA'
        WHEN job_country = 'US' THEN 'US'
        WHEN job_country = 'MX' THEN 'MX'
        WHEN job_country = 'BR' THEN 'BR'
        WHEN job_country = 'CO' THEN 'CO'
        WHEN job_country = 'CL' THEN 'CL'
        WHEN job_country = 'AR' THEN 'AR'
        WHEN job_country = 'IN' THEN 'IN'
        ELSE job_country
      END,
      CASE WHEN job_state IS NOT NULL THEN CONCAT(' - ', job_state) ELSE '' END
    ) AS ubicacion_completa,
    
    CAST(job_is_remote AS BOOLEAN) AS job_is_remote,
    CAST(job_apply_is_direct AS BOOLEAN) AS job_apply_is_direct,
    CAST(job_salary AS DOUBLE) AS job_salary,
    CAST(job_min_salary AS DOUBLE) AS job_min_salary,
    CAST(job_max_salary AS DOUBLE) AS job_max_salary,
    
    CASE 
      WHEN job_min_salary IS NOT NULL AND job_max_salary IS NOT NULL 
        THEN CAST((CAST(job_min_salary AS DOUBLE) + CAST(job_max_salary AS DOUBLE)) / 2 AS DOUBLE)
      WHEN job_salary IS NOT NULL THEN CAST(job_salary AS DOUBLE)
      ELSE NULL
    END AS job_salary_avg,
    
    job_salary_string,
    job_salary_period,
    CAST(job_latitude AS DOUBLE) AS job_latitude,
    CAST(job_longitude AS DOUBLE) AS job_longitude,
    job_apply_link,
    apply_options,
    job_benefits,
    job_benefits_strings,
    job_google_link,
    job_onet_soc,
    job_onet_job_zone,
    
    -- ✅ EXTRACCIÓN DE SENIORITY LEVEL desde job_title
    CASE 
      WHEN LOWER(job_title) LIKE '%senior%' OR LOWER(job_title) LIKE '%sr.%' OR LOWER(job_title) LIKE '%sr %' THEN 'Senior'
      WHEN LOWER(job_title) LIKE '%junior%' OR LOWER(job_title) LIKE '%jr.%' OR LOWER(job_title) LIKE '%jr %' THEN 'Junior'
      WHEN LOWER(job_title) LIKE '%lead%' OR LOWER(job_title) LIKE '%principal%' OR LOWER(job_title) LIKE '%staff%' THEN 'Lead/Staff'
      WHEN LOWER(job_title) LIKE '%entry%' OR LOWER(job_title) LIKE '%trainee%' OR LOWER(job_title) LIKE '%intern%' THEN 'Entry Level'
      WHEN LOWER(job_title) LIKE '%mid%' OR LOWER(job_title) LIKE '%intermediate%' THEN 'Mid-level'
      WHEN LOWER(job_title) LIKE '%chief%' OR LOWER(job_title) LIKE '%director%' OR LOWER(job_title) LIKE '%vp%' OR LOWER(job_title) LIKE '%vice president%' THEN 'Executive'
      ELSE 'No especificado'
    END AS seniority_level,
    
    job_posted_at,
    job_posted_at_datetime_utc,
    job_posted_at_timestamp,
    _rescued_data
    
  FROM deduplicated_data
  WHERE rn = 1
)
SELECT * FROM cleaned_data;

In [0]:
-- Paso 2: MERGE a tabla Silver (preservando registros históricos, evitando duplicados)
-- Solo inserta/actualiza registros basándose en job_id
MERGE INTO prueba_api.silver.jobs_silver AS target
USING (
    SELECT 
        *,
        CURRENT_TIMESTAMP() AS processed_at
    FROM temp_silver_transformed
) AS source
ON target.job_id = source.job_id

-- Si el job_id ya existe, actualizar los datos
WHEN MATCHED THEN
    UPDATE SET *

-- Si el job_id no existe, insertar el nuevo registro
WHEN NOT MATCHED THEN
    INSERT *;

In [0]:
-- Comparar registros entre Bronze y Silver
SELECT 
    'Bronze' AS capa,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT job_id) AS registros_unicos
FROM prueba_api.bronze.jobs_bronze

UNION ALL

SELECT 
    'Silver' AS capa,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT job_id) AS registros_unicos
FROM prueba_api.silver.jobs_silver

UNION ALL

SELECT
    'Diferencia' AS capa,
    (SELECT COUNT(*) FROM prueba_api.bronze.jobs_bronze) - 
    (SELECT COUNT(*) FROM prueba_api.silver.jobs_silver) AS registros_eliminados,
    0 AS placeholder;

In [0]:
DESCRIBE prueba_api.silver.jobs_silver;

In [0]:
SELECT 
    job_id,
    employer_name,
    job_title,
    job_location,
    ubicacion_completa,
    job_is_remote,
    job_employment_type,
    job_salary_avg,
    job_salary_string,
    processed_at
FROM prueba_api.silver.jobs_silver
LIMIT 10;

In [0]:
SELECT 
    employer_name,
    job_title,
    job_salary,
    job_min_salary,
    job_max_salary,
    job_salary_avg,
    job_salary_string,
    job_salary_period
FROM prueba_api.silver.jobs_silver
WHERE job_salary IS NOT NULL 
   OR job_min_salary IS NOT NULL 
   OR job_max_salary IS NOT NULL
ORDER BY job_salary_avg DESC;

In [0]:
-- Análisis de distribución por estado
SELECT 
    job_state AS estado,
    job_country AS pais,
    COUNT(*) AS total_ofertas,
    COUNT(CASE WHEN job_is_remote = true THEN 1 END) AS ofertas_remotas,
    COUNT(CASE WHEN job_is_remote = false THEN 1 END) AS ofertas_presenciales,
    ROUND(COUNT(CASE WHEN job_is_remote = true THEN 1 END) * 100.0 / COUNT(*), 2) AS porcentaje_remoto,
    COUNT(CASE WHEN job_salary_avg IS NOT NULL THEN 1 END) AS ofertas_con_salario,
    ROUND(AVG(job_salary_avg), 0) AS salario_promedio
FROM prueba_api.silver.jobs_silver
GROUP BY job_state, job_country
ORDER BY total_ofertas DESC, estado;

In [0]:
-- Análisis detallado por ciudad
SELECT 
    job_city AS ciudad,
    job_state AS estado,
    ubicacion_completa,
    COUNT(*) AS total_ofertas,
    STRING_AGG(employer_name, ', ') AS empleadores,
    AVG(job_salary_avg) AS salario_promedio_ciudad
FROM prueba_api.silver.jobs_silver
GROUP BY job_city, job_state, ubicacion_completa
ORDER BY total_ofertas DESC, ciudad;

In [0]:
-- Comparación remoto vs presencial
SELECT 
    CASE 
        WHEN job_is_remote = true THEN '🏠 Remoto'
        WHEN job_is_remote = false THEN '🏛️ Presencial'
        ELSE '❓ No especificado'
    END AS modalidad,
    COUNT(*) AS total_ofertas,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM prueba_api.silver.jobs_silver), 2) AS porcentaje,
    COUNT(CASE WHEN job_salary_avg IS NOT NULL THEN 1 END) AS ofertas_con_salario,
    ROUND(AVG(job_salary_avg), 0) AS salario_promedio,
    MIN(job_salary_avg) AS salario_minimo,
    MAX(job_salary_avg) AS salario_maximo
FROM prueba_api.silver.jobs_silver
GROUP BY job_is_remote
ORDER BY total_ofertas DESC;

In [0]:
-- Vista de coordenadas para mapeo
SELECT 
    employer_name,
    job_title,
    job_location,
    ubicacion_completa,
    job_latitude,
    job_longitude,
    job_is_remote,
    job_salary_avg
FROM prueba_api.silver.jobs_silver
WHERE job_latitude IS NOT NULL 
  AND job_longitude IS NOT NULL
ORDER BY job_state, job_city;

In [0]:
-- ============================================
-- 🔧 ENRIQUECIMIENTO DE SITIOS WEB
-- ============================================
-- Crear tabla temporal con mapeo manual de sitios web conocidos
-- para los empleadores más importantes sin sitio web registrado

CREATE OR REPLACE TEMPORARY VIEW employer_website_mapping AS
SELECT * FROM VALUES
    ('BAIRESDEV', 'https://www.bairesdev.com'),
    ('BairesDev', 'https://www.bairesdev.com'),
    ('EY', 'https://www.ey.com'),
    ('Encora Inc.', 'https://www.encora.com'),
    ('AGtec Servicios Informáticos', 'https://www.agtec.cl'),
    ('Core Code io', 'https://www.corecode.io'),
    ('Acid Labs SpA', 'https://www.acidlabs.com'),
    ('Imagemaker', 'https://www.imagemaker.cl'),
    ('PS Grupo Hunting', 'https://www.psgrupo.com'),
    ('2BRAINS', 'https://www.2brains.cl'),
    ('BC Tecnología', 'https://www.bctecnologia.cl'),
    ('BRM S.A.S', 'https://www.brm.com.co'),
    ('BRP', 'https://www.brp.com'),
    ('Bluelight', 'https://www.bluelight.co'),
    ('COPEC S.A.', 'https://www.copec.cl'),
    ('DXC Technology', 'https://www.dxc.com'),
    ('EPAM Systems', 'https://www.epam.com'),
    ('Grupodot', 'https://www.grupodot.com'),
    ('Microsoft', 'https://www.microsoft.com'),
    ('Oracle', 'https://www.oracle.com'),
    ('IBM', 'https://www.ibm.com'),
    ('Accenture', 'https://www.accenture.com'),
    ('AgileEngine', 'https://agileengine.com'),
    ('Blossom', 'https://www.blossom.com'),
    ('Banco Bradesco', 'https://www.bradesco.com.br'),
    ('Boston Consulting Group', 'https://www.bcg.com'),
    ('CI&T', 'https://www.ciandt.com'),
    ('Barclays', 'https://www.barclays.com'),
    ('Bank of Montreal', 'https://www.bmo.com'),
    ('American Software Resources, Inc', 'https://www.asrjobs.com'),
    ('Agentnoon', 'https://www.agentnoon.com')
AS mapping(employer_name, employer_website);

In [0]:
-- ============================================
-- 🔄 ACTUALIZAR TABLA SILVER CON SITIOS WEB
-- ============================================
-- Actualizar employer_website en la tabla Silver usando el mapeo manual
-- Esto asegura que los datos fluyan correctamente a través del pipeline

MERGE INTO prueba_api.silver.jobs_silver AS target
USING employer_website_mapping AS source
ON target.employer_name = source.employer_name
WHEN MATCHED AND target.employer_website IS NULL THEN
  UPDATE SET 
    target.employer_website = source.employer_website;
    
-- Verificar cuántos registros se actualizaron en Silver
SELECT 
    'Registros Silver actualizados' AS metrica,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT employer_name) AS empleadores_unicos
FROM prueba_api.silver.jobs_silver
WHERE employer_website IS NOT NULL
  AND employer_name IN (SELECT employer_name FROM employer_website_mapping);

In [0]:
SELECT * FROM prueba_api.silver.jobs_silver
WHERE employer_name = "BairesDev"